# Análise de Performance - Verilator vs LUT2GPU

Este notebook processa dados de performance de implementações Verilator e LUT2GPU, usando o nome da pasta como dataset e gerando uma linha para cada arquivo CSV.

## Importando bibliotecas necessárias

In [99]:
import os
import pandas as pd
import numpy as np
from IPython.display import display, HTML

## Função para processar um único arquivo CSV

In [100]:
def process_single_csv(filepath, dataset_name):
    """
    Processa um único arquivo CSV e retorna suas informações.
    
    Args:
        filepath (str): Caminho do arquivo CSV
        dataset_name (str): Nome do dataset (pasta)
    
    Returns:
        pd.DataFrame: DataFrame com uma linha de resultados
    """
    try:
        df = pd.read_csv(filepath)
        
        # Identifica métodos disponíveis
        available_methods = df['implementation'].unique()
        
        # Prepara o DataFrame de resultado com valores padrão
        result_df = pd.DataFrame({
            'Timestamp': [df['timestamp'].min()],
            'Dataset': [dataset_name],
            'Inputs': [df['input_size'].unique()[0]],
            'Outputs': [df['output_size'].unique()[0]],
            'Combinations': [2**df['input_size'].unique()[0]],
            'Luts': 5
        })
        
        # Adiciona colunas para cada método
        methods_to_check = ['verilator', 'cuda']

        
        for method in methods_to_check:
            method_data = df[df['implementation'] == method] if method in available_methods else pd.DataFrame()
            
            if not method_data.empty:
                idx_min = method_data['execution_time_ms'].idxmin()
                if method == 'cuda':
                    result_df[f'{method.upper()}_Threads_Block'] = method_data.loc[idx_min, 'threads_per_block']
                    result_df[f'{method.upper()}_Blocks_Grid'] = method_data.loc[idx_min, 'blocks_per_grid']
                    result_df[f'{method.upper()}_Launches'] = method_data.loc[idx_min, 'launches']
                result_df[f'{method.upper()}_ExeTime_ms'] = method_data.loc[idx_min, 'execution_time_ms']
                result_df[f'{method.upper()}_Throughput_sec'] = method_data.loc[idx_min, 'throughput_combinations_per_sec']
            else:
                result_df[f'{method.upper()}_ExeTime_ms'] = '-'
                result_df[f'{method.upper()}_Throughput_sec'] = '-'
                if method == 'cuda':
                    result_df[f'{method.upper()}_Threads_Block'] = '-'
                    result_df[f'{method.upper()}_Blocks_Grid'] = '-'
                    result_df[f'{method.upper()}_Launches'] = '-'
        
        # Calcula diferença se ambos os métodos estiverem presentes e retorna quantas vezes mais rápido é o CUDA
        if 'verilator' in available_methods and 'cuda' in available_methods:
            result_df['CUDA_Speedup'] = result_df['CUDA_Throughput_sec'] / result_df['VERILATOR_Throughput_sec']
        else:
            result_df['CUDA_Speedup'] = '-'
        
        return result_df
    
    except Exception as e:
        print(f"Erro ao processar {filepath}: {e}")
        return None

## Função para processar todas as pastas

In [101]:
def process_all_folders(root_folder):
    """
    Processa todos os arquivos CSV em todas as pastas.
    
    Args:
        root_folder (str): Pasta raiz para busca
    
    Returns:
        pd.DataFrame: DataFrame com todos os resultados
    """
    all_results = []
    
    for dirpath, dirnames, filenames in os.walk(root_folder):
        # Pega o nome da última pasta do caminho como dataset
        dataset_name = os.path.basename(dirpath)
        
        for filename in filenames:
            if filename.endswith('.csv'):
                filepath = os.path.join(dirpath, filename)
                result = process_single_csv(filepath, dataset_name)
                
                if result is not None:
                    all_results.append(result)
    
    if not all_results:
        print("Nenhum arquivo CSV encontrado.")
        return None
    
    # Combina todos os resultados
    return pd.concat(all_results, ignore_index=True)

## Função para salvar resultados

In [102]:
def save_results(df, output_path='performance_summary.csv'):
    """
    Salva resultados em um arquivo CSV e exibe no notebook
    
    Args:
        df (pd.DataFrame): DataFrame com resultados
        output_path (str): Caminho para salvar o arquivo
    """
    if df is not None:
        df.to_csv(output_path, index=False)
        print(f"Resultados salvos em {output_path}")
        display(HTML(df.to_html(index=False)))

## Processando os dados

Agora vamos processar todos os arquivos CSV encontrados e gerar o relatório.

In [103]:
# Processa os arquivos em todas as pastas
root_folder = 'results'
results = process_all_folders(root_folder)

if results is not None:
    save_results(results)

Erro ao processar results/somador_n22/performance_results.csv: index 0 is out of bounds for axis 0 with size 0
Erro ao processar results/verilog_test_c3540/performance_results.csv: index 0 is out of bounds for axis 0 with size 0
Erro ao processar results/verilog_test_c880/performance_results.csv: index 0 is out of bounds for axis 0 with size 0
Resultados salvos em performance_summary.csv


Timestamp,Dataset,Inputs,Outputs,Combinations,Luts,VERILATOR_ExeTime_ms,VERILATOR_Throughput_sec,CUDA_Threads_Block,CUDA_Blocks_Grid,CUDA_Launches,CUDA_ExeTime_ms,CUDA_Throughput_sec,CUDA_Speedup
1.738764e+09,somador_n12,25,13,33554432,5,3124.091,10740541.94,256.0,768.0,1.716667e+02,22.784,3.357800e+11,31262.850323
1.738764e+09,somador_n16,33,17,8589934592,5,785255.888,10939026.02,256.0,768.0,4.369167e+04,6003.229,4.378510e+11,40026.509359
1.738766e+09,somador_n18,37,19,137438953472,5,-,-,256.0,768.0,6.990517e+05,96633.312,4.906842e+11,-
1.738764e+09,somador_n14,29,15,536870912,5,47889.438,11210632.92,256.0,768.0,2.731667e+03,376.391,3.808394e+11,33971.269211
1.738783e+09,somador_n20,41,21,2199023255552,5,-,-,256.0,768.0,1.118481e+07,1638156.625,5.154726e+11,-
1.738803e+09,verilog_test_c1355,41,32,2199023255552,5,-,-,256.0,768.0,1.118481e+07,1899435.000,9.493344e+11,-
1.738805e+09,verilog_test_c17,5,2,32,5,1.706,18755.97,256.0,768.0,1.000158e+00,0.459,7.668845e+05,40.88749
1.738805e+09,verilog_test_c1908,33,25,8589934592,5,-,-,256.0,768.0,4.369167e+04,7069.448,7.922312e+11,-
1.738808e+09,verilog_test_c432,36,7,68719476736,5,-,-,256.0,768.0,3.495263e+05,47940.027,4.687371e+11,-
1.738810e+09,verilog_test_c499,41,32,2199023255552,5,-,-,256.0,768.0,1.118481e+07,1881585.500,8.812055e+11,-


## Visualização dos Resultados

Para visualizar os dados de forma mais detalhada:

In [104]:
if results is not None:
    # Mostra todas as colunas
    pd.set_option('display.max_columns', None)
    display(results)

,Timestamp,Dataset,Inputs,Outputs,Combinations,Luts,VERILATOR_ExeTime_ms,VERILATOR_Throughput_sec,CUDA_Threads_Block,CUDA_Blocks_Grid,CUDA_Launches,CUDA_ExeTime_ms,CUDA_Throughput_sec,CUDA_Speedup
0,1.738764e+09,somador_n12,25,13,33554432,5,3124.091,10740541.94,256.0,768.0,1.716667e+02,22.784,3.357800e+11,31262.850323
1,1.738764e+09,somador_n16,33,17,8589934592,5,785255.888,10939026.02,256.0,768.0,4.369167e+04,6003.229,4.378510e+11,40026.509359
2,1.738766e+09,somador_n18,37,19,137438953472,5,-,-,256.0,768.0,6.990517e+05,96633.312,4.906842e+11,-
3,1.738764e+09,somador_n14,29,15,536870912,5,47889.438,11210632.92,256.0,768.0,2.731667e+03,376.391,3.808394e+11,33971.269211
4,1.738783e+09,somador_n20,41,21,2199023255552,5,-,-,256.0,768.0,1.118481e+07,1638156.625,5.154726e+11,-
5,1.738803e+09,verilog_test_c1355,41,32,2199023255552,5,-,-,256.0,768.0,1.118481e+07,1899435.000,9.493344e+11,-
6,1.738805e+09,verilog_test_c17,5,2,32,5,1.706,18755.97,256.0,768.0,1.000158e+00,0.459,7.668845e+05,40.88749
7,1.738805e+09,verilog_test_c1908,33,25,8589934592,5,-,-,256.0,768.0,4.369167e+04,7069.448,7.922312e+11,-
8,1.738808e+09,verilog_test_c432,36,7,68719476736,5,-,-,256.0,768.0,3.495263e+05,47940.027,4.687371e+11,-
9,1.738810e+09,verilog_test_c499,41,32,2199023255552,5,-,-,256.0,768.0,1.118481e+07,1881585.500,8.812055e+11,-
